## Visualisierung: Volumen, XY- , XZ- , YZ-Schnitte

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import numpy as np
import math
import pandas as pd

In [ ]:
filename = "assets/Voxel_Schneckenstein_II_10x10.nc"
fieldName = "Schneckenstein_II_Prediction_0"

In [ ]:
ds = xr.open_dataset(filename)
data = ds[fieldName]
vol = data.values

vol = np.nan_to_num(vol)
# vmin, vmax = np.percentile(vol, [5, 95])

# Koordinaten:
x_coords = data['x'].values   # z.B. 318000 ... 320000
y_coords = data['y'].values   # z.B. 5587000 ... 5588000
z_coords = data['z'].values   # z.B. 300 ... 800

nz, ny, nx = vol.shape

# Für das 3D-Volume brauchen wir (x, y, z)-Reihenfolge
vol_xyz = np.transpose(vol, (2, 1, 0))  # (x, y, z)

# 3D-Koordinatengitter erzeugen (Plotly erwartet 1D-Arrays)
X, Y, Z = np.meshgrid(x_coords, y_coords, z_coords, indexing='xy')

# Start-Slices (Mitte)
k0  = nz // 2  # z-Index für XY-Schnitt
iy0 = ny // 2  # y-Index für XZ-Schnitt
ix0 = nx // 2  # x-Index für YZ-Schnitt

# ==== Figure mit 4 Panels anlegen ========================================

fig = make_subplots(
    rows=2, cols=2,
    specs=[
        [{'type': 'scene'},    {'type': 'heatmap'}],
        [{'type': 'heatmap'},  {'type': 'heatmap'}],
    ],
    subplot_titles=("3D Volume", "XY-Schnitt", "XZ-Schnitt", "YZ-Schnitt"),
    horizontal_spacing=0.1,
    vertical_spacing=0.1
)

# ---- 1) 3D Volume (oben links) ------------------------------------------

fig.add_trace(go.Volume(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=vol_xyz.flatten(),
    # isomin=vmin,
    # isomax=vmax,
    opacity=0.1,       # Transparenz anpassen (0–1)
    surface_count=15,  # Anzahl Isosurfaces
    colorbar_title=str(data.name),
    coloraxis="coloraxis"
    ),
    row=1, col=1
)

# ---- 1b) Schnitt-Ebene im 3D-Plot (oben links) --------------------------
# Ebene bei z = z_coords[k0], nur zur Visualisierung der Schnittebene

Z_plane0 = np.full((ny, nx), z_coords[k0])

fig.add_trace(
    go.Surface(
        x=x_coords,           # 1D
        y=y_coords,           # 1D
        z=Z_plane0,           # 2D: (ny, nx)
        opacity=0.8,
        showscale=False,      # keine extra Farbskala
        colorscale="Greys",
    ),
    row=1, col=1
)

# ---- 2) XY-Schnitt (oben rechts) ----------------------------------------

fig.add_trace(
    go.Heatmap(
        x=x_coords,
        y=y_coords,
        z=vol[k0, :, :],   # z = z_coords[k0]
        colorbar_title=str(data.name),
        coloraxis="coloraxis"
    ),
    row=1, col=2
)

# ---- 3) XZ-Schnitt (unten links) ----------------------------------------

fig.add_trace(
    go.Heatmap(
        x=x_coords,
        y=z_coords,
        z=vol[:, iy0, :],  # y = y_coords[iy0]
        colorbar_title=str(data.name),
        coloraxis="coloraxis"
    ),
    row=2, col=1
)

# ---- 4) YZ-Schnitt (unten rechts) ---------------------------------------

fig.add_trace(
    go.Heatmap(
        x=y_coords,
        y=z_coords,
        z=vol[:, :, ix0],  # x = x_coords[ix0]
        colorbar_title=str(data.name),
        coloraxis="coloraxis"
    ),
    row=2, col=2
)

# ==== Slider für den XY-Schnitt (Trace 1) ================================

steps = []
for k in range(nz):
    # neue Ebene im 3D-Plot
    Z_plane = np.full((ny, nx), z_coords[k])

    step = dict(
        method="restyle",
        args=[
            {
                # Update für XY-Heatmap (Trace-Index 2)
                "z": [Z_plane, vol[k, :, :]],  # [für Trace 1, für Trace 2]
            },
            [1, 2]   # wir ändern Trace 1 (Surface-Ebene) und Trace 2 (XY-Heatmap)
        ],
        label=f"{z_coords[k]:.2f}"
    )
    steps.append(step)

sliders = [dict(
    active=k0,
    currentvalue={"prefix": "z = "},
    pad={"t": 50},
    steps=steps
)]



fig.update_layout(
    width=2000,
    height=1200,
    sliders=sliders,
    # Achsentitel etwas hübscher
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode='data'
    ),
    coloraxis=dict(
        colorscale="Viridis",          # oder was du magst
        colorbar=dict(
            title=str(data.name),
        )
    )
)

# Optional: Achsentitel für 2D-Subplots
fig.update_xaxes(title_text="X", row=1, col=2)
fig.update_yaxes(title_text="Y", row=1, col=2)

fig.update_xaxes(title_text="X", row=2, col=1)
fig.update_yaxes(title_text="Z", row=2, col=1)

fig.update_xaxes(title_text="Y", row=2, col=2)
fig.update_yaxes(title_text="Z", row=2, col=2)

fig.show()

## Visualisierung Volumen mit Pyvista

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

ds = nc.Dataset('assets/Voxel_Schneckenstein_II_10x10.nc')
data = ds.variables['Schneckenstein_II_Prediction_0'][:]
x = ds.variables['x'][:]
y = ds.variables['y'][:]
z = ds.variables['z'][:]

In [ ]:
import pyvista as pv

# Meshgrid aufbauen
xx, yy, zz = np.meshgrid(x, y, z, indexing='ij')  # (24,10,54)
grid = pv.StructuredGrid(xx, yy, zz)

# Daten transponieren auf (x,y,z) und flatten
values = data.transpose(2, 1, 0).flatten(order='F')
grid['Prediction'] = values

# Interaktiver 3D-Plot
plotter = pv.Plotter()
plotter.add_volume(grid, scalars='Prediction', cmap='RdYlGn', opacity='sigmoid')
plotter.show_axes()
plotter.show()

In [ ]:
iso = grid.contour(isosurfaces=5, scalars='Prediction')
plotter = pv.Plotter()
plotter.add_mesh(iso, cmap='viridis', opacity=0.7)
plotter.show()

## Visualisierung mit Plotly 3D Volume Plot

In [ ]:
import netCDF4 as nc
import numpy as np
import plotly.graph_objects as go

In [ ]:
# Daten laden
ds = nc.Dataset('assets/Voxel_Schneckenstein_II_10x10.nc')
data = ds.variables['Schneckenstein_II_Prediction_0'][:]  # shape: (z=54, y=10, x=24)
x_coords = ds.variables['x'][:]
y_coords = ds.variables['y'][:]
z_coords = ds.variables['z'][:]

In [ ]:
# Meshgrid aufbauen – Reihenfolge muss zu data-shape (z,y,x) passen
Z, Y, X = np.meshgrid(z_coords, y_coords, x_coords, indexing='ij')

# Alles flatten
X_flat = X.flatten()
Y_flat = Y.flatten()
Z_flat = Z.flatten()
val_flat = np.array(data).flatten()

# NaN-Werte maskieren (NetCDF fill values)
mask = ~np.isnan(val_flat)

fig = go.Figure(data=go.Volume(
    x=X_flat[mask],
    y=Y_flat[mask],
    z=Z_flat[mask],
    value=val_flat[mask],
    isomin=np.nanpercentile(val_flat, 10),   # untere 10% ignorieren
    isomax=np.nanpercentile(val_flat, 90),   # obere 10% ignorieren
    opacity=0.1,          # niedrig halten für Durchsicht
    surface_count=20,     # mehr = feineres Rendering, aber langsamer
    colorscale='RdYlGn',
    caps=dict(x_show=False, y_show=False, z_show=False),  # sauberer ohne Caps
))

fig.update_layout(
    title='Schneckenstein II – 3D Volume',
    scene=dict(
        xaxis_title='X (UTM)',
        yaxis_title='Y (UTM)',
        zaxis_title='Z (m)',
        aspectmode='data'  # echte räumliche Proportionen
    )
)

fig.show()